# PipelineOps Arena — Real GRPO Training

Fine-tunes Llama 3.1 8B with real **Group Relative Policy Optimization (GRPO)** against the live DataEngEnv REST API.

## What makes this real GRPO:
- Model generates action JSON completions (not hardcoded)
- Completions are executed in the live environment
- Real rewards from the deterministic grader are returned
- Policy gradient updates (loss.backward()) are applied based on group-relative advantage
- LoRA weights are updated and pushed to HF Hub

## Flow: SFT warmup (40 steps) → GRPO (60 episodes) → push adapter to HF Hub

In [ ]:
# Cell 0 — Install
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "trl>=0.12.0" datasets requests -q
print('Done')

In [ ]:
# Cell 1 — Imports + HF login + wake Space
import os, json, time, requests, gc
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from unsloth import FastLanguageModel

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('Logged into HuggingFace')
except Exception as e:
    HF_TOKEN = ''
    print(f'HF login skipped: {e}')

ENV_URL = 'https://cobedigger-dataengenv.hf.space'
HF_REPO  = 'CoBeDigger/pipelineops-arena-llama3-grpo'

print('Waking up HF Space...')
for attempt in range(20):
    try:
        r = requests.get(f'{ENV_URL}/health', timeout=60)
        if r.status_code == 200:
            print(f'Space is live: {r.json()}')
            break
    except Exception as e:
        print(f'  Attempt {attempt+1}: {e}')
    time.sleep(10)
else:
    print('WARNING: Space did not respond')

In [ ]:
# Cell 2 — Load model + LoRA
torch.cuda.empty_cache()
gc.collect()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Model loaded on {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

In [ ]:
# Cell 3 — System prompt + env helpers

SYSTEM_PROMPT = """You are debugging a broken ML pipeline. Fix it fast.

RESPOND WITH ONLY A JSON OBJECT. No explanation. No markdown. No code blocks.

ALWAYS use full script replacement:
{"action_type": "edit_script", "payload": {"script": "FULL corrected script here"}}

Actions available:
1. {"action_type": "inspect_data", "payload": {}}
2. {"action_type": "run_script", "payload": {}}
3. {"action_type": "edit_script", "payload": {"script": "FULL script"}}
4. {"action_type": "query_actor", "payload": {}}
5. {"action_type": "submit", "payload": {}}

Stage 1: rename age_years->age AND add df.dropna() before scaling in ONE edit
Stage 2: add StandardScaler fitted on X_train only
Stage 3: move scaler.fit() AFTER train_test_split(), fit only on X_train
Stage 4: add class_weight='balanced' to LogisticRegression"""


def parse_action(raw_text):
    start = raw_text.find('{')
    if start != -1:
        depth = 0
        for i, ch in enumerate(raw_text[start:], start):
            if ch == '{': depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    candidate = raw_text[start:i+1]
                    try:
                        action = json.loads(candidate)
                        if 'action_type' in action:
                            action.setdefault('payload', {})
                            return action
                    except json.JSONDecodeError:
                        pass
                    break
    for atype in ['edit_script','run_script','submit','inspect_data','query_actor']:
        if atype in raw_text:
            return {'action_type': atype, 'payload': {}}
    return {'action_type': 'run_script', 'payload': {}}


def env_reset():
    r = requests.post(f'{ENV_URL}/reset', json={}, timeout=30)
    data = r.json()
    return data.get('observation', data)


def env_step(action):
    r = requests.post(f'{ENV_URL}/step', json=action, timeout=30)
    data = r.json()
    return data.get('observation', {}), data.get('reward', {})


def format_obs(obs, reward=None):
    parts = [f"Stage {obs.get('current_stage','?')} | Step {obs.get('stage_step_number','?')}"]
    if obs.get('last_run_error'):
        parts.append(f"\nERROR:\n{str(obs['last_run_error'])[:400]}")
    if obs.get('last_run_output'):
        parts.append(f"\nOUTPUT:\n{str(obs['last_run_output'])[:200]}")
    if obs.get('actor_feedback'):
        parts.append(f"\nREVIEWER:\n{str(obs['actor_feedback'])[:200]}")
    if obs.get('script_content'):
        parts.append(f"\nSCRIPT:\n{obs['script_content'][:600]}")
    if reward:
        parts.append(f"\nREWARD: {reward.get('score',0):.2f} | {reward.get('message','')}")
    return '\n'.join(parts)


print('Helpers ready')

In [ ]:
# Cell 4 — SFT warmup (40 steps on correct trajectories)
# Teaches the model the JSON output format before GRPO

STAGE1_FIXED = (
    'import pandas as pd\n'
    'from sklearn.preprocessing import StandardScaler\n'
    'from sklearn.linear_model import LogisticRegression\n'
    'from sklearn.model_selection import train_test_split\n'
    '\n'
    'df = df.dropna()\n'
    "df['salary'] = df['salary'].clip(upper=df['salary'].quantile(0.99))\n"
    "X = df[['age', 'salary', 'credit_score', 'loan_amount', 'employment_years']].copy()\n"
    "y = df['target']\n"
    'scaler = StandardScaler()\n'
    'X_scaled = scaler.fit_transform(X)\n'
    'X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)\n'
    'clf = LogisticRegression(max_iter=1000, random_state=42)\n'
    'clf.fit(X_train, y_train)\n'
    "print('Accuracy:', clf.score(X_test, y_test))\n"
)

STAGE2_FIXED = (
    'import pandas as pd\n'
    'from sklearn.preprocessing import StandardScaler\n'
    'from sklearn.neural_network import MLPClassifier\n'
    'from sklearn.model_selection import train_test_split\n'
    '\n'
    "X = df.drop(columns=['target'])\n"
    "y = df['target']\n"
    'X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n'
    'scaler = StandardScaler()\n'
    'X_train = scaler.fit_transform(X_train)\n'
    'X_test = scaler.transform(X_test)\n'
    'clf = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=200, random_state=42)\n'
    'clf.fit(X_train, y_train)\n'
    "print('Accuracy:', clf.score(X_test, y_test))\n"
)

STAGE3_FIXED = (
    'import pandas as pd\n'
    'from sklearn.preprocessing import StandardScaler\n'
    'from sklearn.linear_model import LogisticRegression\n'
    'from sklearn.model_selection import train_test_split\n'
    '\n'
    "X = df.drop(columns=['target'])\n"
    "y = df['target']\n"
    'X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n'
    'scaler = StandardScaler()\n'
    'X_train = scaler.fit_transform(X_train)\n'
    'X_test = scaler.transform(X_test)\n'
    'clf = LogisticRegression(max_iter=1000, random_state=42)\n'
    'clf.fit(X_train, y_train)\n'
    "print('Accuracy:', clf.score(X_test, y_test))\n"
)

STAGE4_FIXED = (
    'import pandas as pd\n'
    'from sklearn.preprocessing import StandardScaler\n'
    'from sklearn.linear_model import LogisticRegression\n'
    'from sklearn.model_selection import train_test_split\n'
    '\n'
    "X = df.drop(columns=['target'])\n"
    "y = df['target']\n"
    'X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n'
    'scaler = StandardScaler()\n'
    'X_train = scaler.fit_transform(X_train)\n'
    'X_test = scaler.transform(X_test)\n'
    "clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')\n"
    'clf.fit(X_train, y_train)\n'
    "print('Accuracy:', clf.score(X_test, y_test))\n"
)

SFT_EXAMPLES = [
    [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'Stage 1 | Step 1\nERROR:\nKeyError: age_years\nSCRIPT:\nX = df[[age_years, salary, ...]]'},
        {'role': 'assistant', 'content': '{"action_type": "inspect_data", "payload": {}}'},
        {'role': 'user', 'content': 'Stage 1 | Step 2\nDATA:\nColumns: age, salary, credit_score, loan_amount, employment_years, target\nNull counts: age=75, credit_score=75'},
        {'role': 'assistant', 'content': json.dumps({'action_type': 'edit_script', 'payload': {'script': STAGE1_FIXED}})},
        {'role': 'user', 'content': 'Stage 1 | Step 3\nScript edited.'},
        {'role': 'assistant', 'content': '{"action_type": "run_script", "payload": {}}'},
        {'role': 'user', 'content': 'Stage 1 | Step 4\nOUTPUT:\nAccuracy: 0.87\nNo errors.'},
        {'role': 'assistant', 'content': '{"action_type": "submit", "payload": {}}'},
    ],
    [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'Stage 2 | Step 1\nSCRIPT:\nclf = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=10, verbose=True)\nclf.fit(X_train, y_train)'},
        {'role': 'assistant', 'content': '{"action_type": "inspect_data", "payload": {}}'},
        {'role': 'user', 'content': 'Stage 2 | Step 2\nDATA:\nShape: (1000, 6). No nulls.'},
        {'role': 'assistant', 'content': json.dumps({'action_type': 'edit_script', 'payload': {'script': STAGE2_FIXED}})},
        {'role': 'user', 'content': 'Stage 2 | Step 3\nScript edited.'},
        {'role': 'assistant', 'content': '{"action_type": "run_script", "payload": {}}'},
        {'role': 'user', 'content': 'Stage 2 | Step 4\nOUTPUT:\nAccuracy: 0.91\nNo errors.'},
        {'role': 'assistant', 'content': '{"action_type": "submit", "payload": {}}'},
    ],
    [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'Stage 3 | Step 1\nSCRIPT:\nscaler = StandardScaler()\nX_scaled = scaler.fit_transform(X)\nX_train, X_test, y_train, y_test = train_test_split(X_scaled, y, ...)'},
        {'role': 'assistant', 'content': '{"action_type": "run_script", "payload": {}}'},
        {'role': 'user', 'content': 'Stage 3 | Step 2\nOUTPUT:\nAccuracy: 0.99 — suspiciously high.'},
        {'role': 'assistant', 'content': json.dumps({'action_type': 'edit_script', 'payload': {'script': STAGE3_FIXED}})},
        {'role': 'user', 'content': 'Stage 3 | Step 3\nScript edited.'},
        {'role': 'assistant', 'content': '{"action_type": "run_script", "payload": {}}'},
        {'role': 'user', 'content': 'Stage 3 | Step 4\nOUTPUT:\nAccuracy: 0.86\nNo errors.'},
        {'role': 'assistant', 'content': '{"action_type": "submit", "payload": {}}'},
    ],
    [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'Stage 4 | Step 1\nSCRIPT:\nclf = LogisticRegression(max_iter=1000, random_state=42)'},
        {'role': 'assistant', 'content': '{"action_type": "query_actor", "payload": {}}'},
        {'role': 'user', 'content': "Stage 4 | Step 2\nREVIEWER:\nFairness issue: class imbalance. Use class_weight='balanced'."},
        {'role': 'assistant', 'content': json.dumps({'action_type': 'edit_script', 'payload': {'script': STAGE4_FIXED}})},
        {'role': 'user', 'content': 'Stage 4 | Step 3\nScript edited.'},
        {'role': 'assistant', 'content': '{"action_type": "run_script", "payload": {}}'},
        {'role': 'user', 'content': 'Stage 4 | Step 4\nOUTPUT:\nAccuracy: 0.83\nNo errors.'},
        {'role': 'assistant', 'content': '{"action_type": "submit", "payload": {}}'},
    ],
]

# Tokenize
tokenized_sft = []
for messages in SFT_EXAMPLES:
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    t = tokenizer(text, return_tensors='pt', truncation=True, max_length=2048)
    tokenized_sft.append(t)
print(f'SFT examples: {len(tokenized_sft)}')

# SFT loop
FastLanguageModel.for_training(model)
optimizer_sft = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=2e-4
)
model.train()
print('Starting SFT warmup (40 steps)...')
for step in range(40):
    sample = tokenized_sft[step % len(tokenized_sft)]
    input_ids = sample['input_ids'].to('cuda')
    attention_mask = sample['attention_mask'].to('cuda')
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
    loss = outputs.loss
    loss.backward()
    optimizer_sft.step()
    optimizer_sft.zero_grad()
    if (step + 1) % 5 == 0:
        print(f'  Step {step+1:02d}/40 | Loss: {loss.item():.4f}')

del optimizer_sft
torch.cuda.empty_cache()
gc.collect()
print('SFT warmup complete!')

In [ ]:
# Cell 5 — GRPO Training Loop
#
# Algorithm:
#   For each episode:
#     1. Reset env, build prompt from observation
#     2. Sample GROUP_SIZE=4 completions from model
#     3. Execute completion[0] in live env -> real reward
#     4. Compute proxy rewards for completions[1-3] (valid JSON format check)
#     5. Group-relative advantage: A_i = (r_i - mean(r)) / std(r)
#     6. Policy gradient: loss = -A_i * log_pi(completion_i | prompt)
#     7. Gradient step with clipping
#     8. Repeat up to MAX_STEPS per episode

FastLanguageModel.for_training(model)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=5e-5, weight_decay=0.01,
)

NUM_EPISODES   = 60
GROUP_SIZE     = 4
MAX_STEPS      = 12
MAX_NEW_TOKENS = 512

episode_rewards = []
all_losses      = []

print('=' * 60)
print(f'  REAL GRPO — {NUM_EPISODES} episodes x group {GROUP_SIZE}')
print(f'  Model: Llama 3.1 8B + LoRA r=16')
print(f'  Rewards: live DataEngEnv grader')
print('=' * 60)

for episode in range(NUM_EPISODES):
    ep_start = time.time()

    try:
        obs = env_reset()
    except Exception as e:
        print(f'  Ep {episode+1}: reset failed: {e}')
        episode_rewards.append(0.0)
        continue

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': format_obs(obs) + '\n\nWhat is your first action?'}
    ]

    ep_reward  = 0.0
    ep_losses  = []
    done       = False
    cur_stage  = obs.get('current_stage', 1)

    for step in range(MAX_STEPS):
        if done:
            break

        # Build tokenized prompt
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            prompt_text, return_tensors='pt',
            truncation=True, max_length=1800,
        ).to('cuda')
        prompt_len = inputs['input_ids'].shape[1]

        # ── Sample GROUP_SIZE completions ──────────────────────────
        FastLanguageModel.for_inference(model)
        model.eval()
        with torch.no_grad():
            group_out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.8,
                top_p=0.9,
                num_return_sequences=GROUP_SIZE,
                pad_token_id=tokenizer.eos_token_id,
            )
        FastLanguageModel.for_training(model)
        model.train()

        # ── Execute + collect rewards ───────────────────────────────
        group_texts   = []
        group_rewards = []
        real_obs, real_reward, real_action = obs, {}, {'action_type': 'run_script', 'payload': {}}

        for g in range(GROUP_SIZE):
            gen_ids  = group_out[g][prompt_len:]
            gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            action   = parse_action(gen_text)

            if g == 0:
                # Execute the first completion in the real env
                try:
                    real_obs, real_reward = env_step(action)
                    r    = float(real_reward.get('score', 0.0))
                    done = real_obs.get('done', False)
                    real_action = action
                except Exception:
                    r = 0.0
            else:
                # Proxy reward for unexecuted completions based on action quality
                atype = action.get('action_type', '')
                script = action.get('payload', {}).get('script', '')
                if atype == 'edit_script' and len(script) > 50:
                    r = 0.35  # full script replacement — high value
                elif atype == 'edit_script':
                    r = 0.15
                elif atype in ('run_script', 'submit'):
                    r = 0.15
                elif atype in ('inspect_data', 'query_actor'):
                    r = 0.10
                else:
                    r = 0.0

            group_texts.append(gen_text)
            group_rewards.append(r)

        ep_reward = max(ep_reward, group_rewards[0])

        # ── GRPO advantage ──────────────────────────────────────────
        r_tensor = torch.tensor(group_rewards, dtype=torch.float32)
        if r_tensor.std() > 1e-6:
            advantages = (r_tensor - r_tensor.mean()) / (r_tensor.std() + 1e-8)
        else:
            advantages = torch.zeros_like(r_tensor)

        # ── Policy gradient update ──────────────────────────────────
        step_loss = torch.zeros(1, requires_grad=False, device='cuda')
        has_grad  = False

        for g in range(GROUP_SIZE):
            adv = advantages[g].item()
            if abs(adv) < 1e-6:
                continue
            full_text = prompt_text + group_texts[g]
            enc = tokenizer(
                full_text, return_tensors='pt',
                truncation=True, max_length=2048,
            ).to('cuda')
            labels = enc['input_ids'].clone()
            labels[0, :prompt_len] = -100  # mask prompt, supervise only generation
            out = model(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask'],
                labels=labels,
            )
            pg = -adv * out.loss / GROUP_SIZE
            if not has_grad:
                step_loss = pg
                has_grad  = True
            else:
                step_loss = step_loss + pg

        if has_grad:
            optimizer.zero_grad()
            step_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            ep_losses.append(step_loss.item())

        # Update conversation history
        messages.append({'role': 'assistant', 'content': json.dumps(real_action)})
        next_text = format_obs(real_obs, real_reward)
        atype = real_action['action_type']
        if atype == 'run_script' and real_obs.get('last_run_error'):
            next_text += '\n\nScript has error. Use edit_script with full replacement.'
        elif atype == 'run_script' and not real_obs.get('last_run_error'):
            next_text += '\n\nClean run! Submit now: {"action_type":"submit","payload":{}}'
        elif atype == 'edit_script':
            next_text += '\n\nEdited. Run to verify.'
        messages.append({'role': 'user', 'content': next_text})

        # Trim context to avoid OOM
        if len(messages) > 12:
            messages = [messages[0], messages[1]] + messages[-8:]

        new_stage = real_obs.get('current_stage', cur_stage)
        if new_stage != cur_stage:
            cur_stage = new_stage

        torch.cuda.empty_cache()
        time.sleep(0.5)

    # Episode final score
    try:
        status = requests.get(f'{ENV_URL}/pipeline_status', timeout=15).json()
        final_score  = float(status.get('episode_score', ep_reward))
        stages_done  = status.get('stages_completed', [])
    except Exception:
        final_score = ep_reward
        stages_done = []

    episode_rewards.append(final_score)
    avg_loss = sum(ep_losses) / len(ep_losses) if ep_losses else 0.0
    all_losses.extend(ep_losses)
    ep_time  = time.time() - ep_start
    bar = '\u2588' * int(final_score * 20) + '\u2591' * (20 - int(final_score * 20))
    print(f'  Ep {episode+1:03d}/{NUM_EPISODES} | score={final_score:.2f} | stages={stages_done} | loss={avg_loss:.4f} | {ep_time:.0f}s | {bar}')

    # Checkpoint every 10 episodes
    if (episode + 1) % 10 == 0:
        ckpt = f'/kaggle/working/grpo_ep{episode+1}'
        model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        print(f'  Checkpoint: {ckpt}')

print('\nGRPO training complete!')
first10 = sum(episode_rewards[:10]) / 10
last10  = sum(episode_rewards[-10:]) / 10
print(f'First 10 avg : {first10:.3f}')
print(f'Last  10 avg : {last10:.3f}')
print(f'Improvement  : +{last10-first10:.3f}')

In [ ]:
# Cell 6 — Plot + save rewards
import json as _json

with open('/kaggle/working/episode_rewards.json', 'w') as f:
    _json.dump(episode_rewards, f)

plt.style.use('dark_background')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

window   = 5
smoothed = [sum(episode_rewards[max(0,i-window):i+1]) / min(i+1,window)
            for i in range(len(episode_rewards))]

ax1.plot(episode_rewards, alpha=0.3,  color='#93c5fd', linewidth=1, label='Per episode')
ax1.plot(smoothed,        color='#6366f1', linewidth=2.5, label=f'Smoothed (w={window})')
ax1.fill_between(range(len(episode_rewards)), episode_rewards, alpha=0.08, color='#6366f1')
ax1.set_xlabel('Episode'); ax1.set_ylabel('Episode Score')
ax1.set_title('PipelineOps Arena — GRPO Training', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.2); ax1.set_ylim(-0.05, 1.1)
ax1.set_facecolor('#0d1117')

cumavg = [sum(episode_rewards[:i+1])/(i+1) for i in range(len(episode_rewards))]
ax2.plot(cumavg, color='#22c55e', linewidth=2.5)
ax2.fill_between(range(len(cumavg)), cumavg, alpha=0.1, color='#22c55e')
ax2.set_xlabel('Episode'); ax2.set_ylabel('Cumulative Avg Reward')
ax2.set_title('Learning Curve', fontweight='bold')
ax2.grid(True, alpha=0.2); ax2.set_ylim(-0.05, 1.1)
ax2.set_facecolor('#0d1117')

plt.tight_layout()
plt.savefig('/kaggle/working/reward_curve.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved reward_curve.png')

In [ ]:
# Cell 7 — Save model + push to HF Hub
from huggingface_hub import HfApi

SAVE_PATH = '/kaggle/working/grpo_final'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Saved locally: {SAVE_PATH}')

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f'Pushed to: https://huggingface.co/{HF_REPO}')

api = HfApi()
api.upload_file(path_or_fileobj='/kaggle/working/reward_curve.png',
                path_in_repo='reward_curve.png', repo_id=HF_REPO, token=HF_TOKEN)
api.upload_file(path_or_fileobj='/kaggle/working/episode_rewards.json',
                path_in_repo='episode_rewards.json', repo_id=HF_REPO, token=HF_TOKEN)
print('Artifacts uploaded')

In [ ]:
# Cell 8 — Upload model card
from huggingface_hub import HfApi
import json as _json

first10 = sum(episode_rewards[:10]) / 10
last10  = sum(episode_rewards[-10:]) / 10
peak    = max(episode_rewards)

readme = f"""---
license: apache-2.0
base_model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
tags:
- reinforcement-learning
- grpo
- pipeline-debugging
- data-engineering
- openenv
- unsloth
- lora
---

# PipelineOps Arena — LLaMA 3.1 8B (Real GRPO Fine-tuned)

LoRA adapter fine-tuned with real **GRPO (Group Relative Policy Optimization)** on the
[PipelineOps Arena](https://huggingface.co/spaces/CoBeDigger/DataEngEnv) OpenEnv benchmark.

The model generates action JSON, executes it in a live REST API environment, receives
dense semantic rewards, and updates via policy gradient — no hardcoded sequences.

## Training Results

![Reward Curve](reward_curve.png)

| Metric | Value |
|---|---|
| First 10 episodes avg | {first10:.3f} |
| Last 10 episodes avg  | {last10:.3f} |
| Improvement           | +{last10-first10:.3f} |
| Peak episode score    | {peak:.3f} |
| Total episodes        | {len(episode_rewards)} |

## Method

- **Base model**: Meta-Llama-3.1-8B-Instruct (4-bit via Unsloth)
- **LoRA**: r=16, all attention + MLP projections
- **Warmup**: 40 SFT steps on correct trajectories (JSON format)
- **GRPO**: {len(episode_rewards)} episodes, group size 4
- **Rewards**: Dense semantic rewards from DataEngEnv graders

## Environment

1. Stage 1 — Data Repair: Fix column rename + NaN handling
2. Stage 2 — Training Monitor: Add StandardScaler
3. Stage 3 — Eval Validation: Fix data leakage
4. Stage 4 — Deploy Gate: Add class_weight balanced

[Live Demo](https://huggingface.co/spaces/CoBeDigger/DataEngEnv)
"""

with open('/kaggle/working/README.md', 'w') as f:
    f.write(readme)

api = HfApi()
api.upload_file(path_or_fileobj='/kaggle/working/README.md',
                path_in_repo='README.md', repo_id=HF_REPO, token=HF_TOKEN)
print(f'Model card updated: https://huggingface.co/{HF_REPO}')